# Whisper testing

This notebook is used to compare Whisper efficiency and accuracy based on the model used (small, base, tiny etc) and other parameters. During end-to-end testing of the application pipeline, latency is prohibitive, with Whisper being one of the components contributing the most to it with an average latency of 2.919 seconds per sentence passed.

The dataset used to determine the bast Whisper variant is espnet / yodas-granary from Hugging Face (https://huggingface.co/datasets/espnet/yodas-granary). This is a modified version of the larger nvidia/Granary dataset, specifically designed for ASR across 23 languages. The transcriptions of the audio clips have been derived faster-whisper-large-v3 model. Given the size and complexity of this model, most transcriptions are expected to be correct. However, in the future the accuracy of the dataset should be examined. Given that for now the application only supports English, the dataset is filtered to extract the entries in English. If we decide to extend to other languages later, this dataset will be helpful in testing Whisper in other languages as well.

## Loading

In [1]:
!pip install --upgrade pip
!pip install faster-whisper
!pip install pandas
!pip install numpy
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 51.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 128.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [faster-whisper]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [2]:
from datasets import load_dataset
import torch
from faster_whisper import WhisperModel
import pandas as pd
import jiwer
import time
import numpy as np

In [3]:
ds = load_dataset("espnet/yodas-granary", "English", streaming = True)            # Load only english
print(ds.column_names)

Resolving data files:   0%|          | 0/18496 [00:00<?, ?it/s]

{'asr_only': ['utt_id', 'audio', 'duration', 'lang', 'task', 'text', 'translation_en', 'original_audio_id', 'original_audio_offset']}


## Testing


To test the affect of changing the Whisper model variant in accuracy and performance the following models are used "small.en", "distil-small.en", "tiny.en", "base.en", "distil-medium.en", "large-v1", "distil-large-v2", "distil-large-v3", "large-v3-turbo", and "turbo".

On the one hand, distil-small, distil-large and distil-medium were trained by freezing the encoder layers and using only the first and the last decoder layers from the original whisper model, discarding the rest. According to the creators, distiled Whisper is 6 times faster, 49% smaller and performs within 1% WER of the original model. Moreover, it is more robust to noise and hallucinations. On the other hand, turbo whisper is inspired from distil whisper and it is an optimization of whipser-large-v3. It hs only 4 decoder layers instead of 32, but instead of using distillation, the model is trained for two more epochs over the same amount of data as large-v3. To evaluate the models above, 2 metrics are used:

* WER : calculates substitutions, deletions and insertions on word level measuring the accuracy of the model
* RTFx (Inverse real time factor) : ratio of duration divided by processing time measuring the latency of the model

### Assumption

The assumption used during testing is that the audio clips are shorter than 30 seconds, which is the maximum receptive field of the Whisper models. It is crucial that the possibility of larger audio clips provided by the user is considered. In this case, chunking is needed to process the audio clips. In the first round of testing with no quantization, the precision used is float16.

### Optimizations

In [6]:
# Picking GPU device if available
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device : {device} \n")

Device : cuda 



In [5]:
model_ids = ["small.en", "distil-small.en", "tiny.en", "base.en", "distil-medium.en", "large-v1", "distil-large-v2", "distil-large-v3", "large-v3-turbo", "turbo"]


In [6]:
# Transformations for WER
results = pd.DataFrame(columns = ['model', 'wer', 'rtf', 'time'])
for id in model_ids:
  print(f"Model : {id} \n")
  df = iter(ds['asr_only'])
  rtfList = np.zeros(100)
  werList = np.zeros(100)
  timeList = np.zeros(100)

  # Loading model
  model = WhisperModel(
      model_size_or_path = id,
      device = device,
      compute_type = "float16",               # Tuned parameter
      use_auth_token = True,
      # flash_attention = True,               # Tuned parameter
      # tensor_parallel = True,               # Tuned parameter
  )

  for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5,                          # Tuned parameter
        # vad_filter = True                     # Tuned parameter
        # condition_on_previous_text = False    # Tuned parameter
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

  averageWER = np.mean(werList)
  averageRTF = np.mean(rtfList)
  averageTime = np.mean(timeList)

  print(f"WER for {id} : {averageWER} \n")
  print(f"RTF for {id} : {averageRTF} \n")
  print(f"Duration for {id} : {averageTime} \n")

  # Update pandas dataframe of results
  results = pd.concat([pd.DataFrame([[id, averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)
  print(results)



Model : small.en 

WER for small.en : 0.16998369118722534 

RTF for small.en : 25.161246718114338 

Duration for small.en : 0.3469259020500022 

      model       wer        rtf      time
0  small.en  0.169984  25.161247  0.346926
Model : distil-small.en 



/tmp/ipykernel_685/1386688782.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([pd.DataFrame([[id, averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


WER for distil-small.en : 0.22683427872526873 

RTF for distil-small.en : 32.12734594741425 

Duration for distil-small.en : 0.32690427234000025 

             model       wer        rtf      time
0  distil-small.en  0.226834  32.127346  0.326904
1         small.en  0.169984  25.161247  0.346926
Model : tiny.en 

WER for tiny.en : 0.2633371508286832 

RTF for tiny.en : 37.83802078491967 

Duration for tiny.en : 0.23248521393000032 

             model       wer        rtf      time
0          tiny.en  0.263337  37.838021  0.232485
1  distil-small.en  0.226834  32.127346  0.326904
2         small.en  0.169984  25.161247  0.346926
Model : base.en 

WER for base.en : 0.21666484221219634 

RTF for base.en : 35.251954205556835 

Duration for base.en : 0.26907500360999675 

             model       wer        rtf      time
0          base.en  0.216665  35.251954  0.269075
1          tiny.en  0.263337  37.838021  0.232485
2  distil-small.en  0.226834  32.127346  0.326904
3         small.en  0

## Parameter optimizatioon

After the optimal model has been found from the above with the optimization techniques outlined, several parameters of the model are tweaked to deduce whether they improve latency significantly. Those are:
1. disabled condition on previous text to reduce latency
2. enabled vad filter for reduced latency and address hallucination
3. beam size changed to 1 to reduce latency
4. flash attention enabled
5. tensor parallelism enabled
6. quantization to int8

The above are used on the base Whisper model.

In [7]:
results = pd.DataFrame(columns = ['technique', 'wer', 'rtf', 'time'])
print(results)

Empty DataFrame
Columns: [technique, wer, rtf, time]
Index: []


In [8]:
# Quantization
df = iter(ds['asr_only'])
rtfList = np.zeros(100)
werList = np.zeros(100)
timeList = np.zeros(100)

model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "int8",
    use_auth_token = True
)

for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for quantization : {averageWER} \n")
print(f"RTF for quantization : {averageRTF} \n")
print(f"Duration for quantization : {averageTime} \n")

results = pd.concat([pd.DataFrame([['quantization', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


WER for quantization : 0.2211716517851553 

RTF for quantization : 29.677978307426397 

Duration for quantization : 0.3478350839799896 



/tmp/ipykernel_12554/2186659959.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([pd.DataFrame([['quantization', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


In [ ]:
# Flash attention and parallelism
df = iter(ds['asr_only'])
rtfList = np.zeros(100)
werList = np.zeros(100)
timeList = np.zeros(100)

model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "float16",
    use_auth_token = True,
    flash_attention = True,
    tensor_parallel = True
)

for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for flash attention : {averageWER} \n")
print(f"RTF for flash attention : {averageRTF} \n")
print(f"Duration for flash attention : {averageTime} \n")

results = pd.concat([pd.DataFrame([['flash', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


In [9]:
model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "float16",
    use_auth_token = True,
)
df = iter(ds['asr_only'])


# Beam size
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 1
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for beam size : {averageWER} \n")
print(f"RTF for beam size : {averageRTF} \n")
print(f"Duration for beam size : {averageTime} \n")

results = pd.concat([pd.DataFrame([['beam', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

# VAD filter
df = iter(ds['asr_only'])
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5,
        vad_filter = True
    )
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    result = " ".join([segment.text for segment in segments])
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for vad : {averageWER} \n")
print(f"RTF for vad : {averageRTF} \n")
print(f"Duration for vad : {averageTime} \n")

results = pd.concat([pd.DataFrame([['vad', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

# Previous text
df = iter(ds['asr_only'])
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5,
        condition_on_previous_text = False
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER for previous text : {averageWER} \n")
print(f"RTF for previous text : {averageRTF} \n")
print(f"Duration for previous text : {averageTime} \n")

results = pd.concat([pd.DataFrame([['previous', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

# Benchamrk
df = iter(ds['asr_only'])
for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 5
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"WER (benchmark) : {averageWER} \n")
print(f"RTF (benchmark) : {averageRTF} \n")
print(f"Duration (benchmark) : {averageTime} \n")
results = pd.concat([pd.DataFrame([['Plain', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)

WER for beam size : 0.2105919905155105 

RTF for beam size : 63.306266426875396 

Duration for beam size : 0.14424854264997522 

WER for vad : 0.23840414595418705 

RTF for vad : 208.14549893396662 

Duration for vad : 0.04688725747003446 

WER for previous text : 0.22263109916952473 

RTF for previous text : 34.77963053222191 

Duration for previous text : 0.2730087147699578 

WER (benchmark) : 0.21666484221219634 

RTF (benchmark) : 34.79788499171782 

Duration (benchmark) : 0.27394447669000327 



In [10]:
# Combined (apart from flash attention) for base
df = iter(ds['asr_only'])
rtfList = np.zeros(100)
werList = np.zeros(100)
timeList = np.zeros(100)

model = WhisperModel(
    model_size_or_path = "base.en",
    device = device,
    compute_type = "float16",
    use_auth_token = True,
    tensor_parallel = True
)

for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 1,
        vad_filter = True,
        condition_on_previous_text = False
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"Combined (base) : {averageWER} \n")
print(f"Combined (base) : {averageRTF} \n")
print(f"Combined (base) : {averageTime} \n")

results = pd.concat([pd.DataFrame([['combined(base)', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)


Combined (base) : 0.2381056976025979 

Combined (base) : 51.84104939904458 

Combined (base) : 0.18309090932993968 



In [ ]:
# Combined (apart from flash attention) for tiny
df = iter(ds['asr_only'])
rtfList = np.zeros(100)
werList = np.zeros(100)
timeList = np.zeros(100)

model = WhisperModel(
    model_size_or_path = "tiny.en",
    device = device,
    compute_type = "float16",
    use_auth_token = True,
    tensor_parallel = True
)

for index in range(100):
    # Inference
    item = next(df)
    start = time.perf_counter()
    segments, _ = model.transcribe(
        item["audio"]["array"],
        beam_size = 1,
        vad_filter = True,
        condition_on_previous_text = False
    )
    result = " ".join([segment.text for segment in segments])
    end = time.perf_counter()
    duration = end - start

    # World error rate
    reference = item["text"]
    wer = jiwer.wer(
      reference,
      result
    )

    # Inverse real time factor
    rtf = item["duration"] / duration

    # Storing results
    werList[index] = wer
    rtfList[index] = rtf
    timeList[index] = duration

averageWER = np.mean(werList)
averageRTF = np.mean(rtfList)
averageTime = np.mean(timeList)

print(f"Combined (tiny) : {averageWER} \n")
print(f"Combined (tiny) : {averageRTF} \n")
print(f"Combined (tiny) : {averageTime} \n")

results = pd.concat([pd.DataFrame([['combined(tiny)', averageWER, averageRTF, averageTime]], columns=results.columns), results], ignore_index=True)